In [83]:
# Quarterly depedency graph

In [84]:
import os
os.getcwd()

'C:\\Users\\ugne.keliauskaite\\Bruegel\\Research - 2021-11 European natural gas imports\\Data'

In [85]:
#Share_point = r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Data' # Gio
Share_point = r'C:\Users\ugne.keliauskaite\Bruegel\Research - 2021-11 European natural gas imports\Data' # Ugne
os.chdir(Share_point)

In [86]:
import json
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.dates import DateFormatter
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns

In [87]:
# import ENTSOG pipeline data
entsog = pd.read_csv(r'Imports\EU27\df1.csv')
del entsog['dates.1']
entsog = entsog.set_index(pd.DatetimeIndex(entsog['dates']))
del entsog['dates']

In [88]:
entsog

,values,direction,operator,location,label,country,exportcountry,IEAname,aggregation,aggregation2,aggregation3
dates,,,,,,,,,,,
2015-01-01,81199128.0,entry,FR-TSO-0003,LNG-00029,Fos (Tonkin/Cavaou),FR,Liquefied Natural Gas,Fos sur Mer,LNG,France,LNG
2015-01-02,81890058.0,entry,FR-TSO-0003,LNG-00029,Fos (Tonkin/Cavaou),FR,Liquefied Natural Gas,Fos sur Mer,LNG,France,LNG
2015-01-03,80995782.0,entry,FR-TSO-0003,LNG-00029,Fos (Tonkin/Cavaou),FR,Liquefied Natural Gas,Fos sur Mer,LNG,France,LNG
2015-01-04,81223971.0,entry,FR-TSO-0003,LNG-00029,Fos (Tonkin/Cavaou),FR,Liquefied Natural Gas,Fos sur Mer,LNG,France,LNG
2015-01-05,96945810.0,entry,FR-TSO-0003,LNG-00029,Fos (Tonkin/Cavaou),FR,Liquefied Natural Gas,Fos sur Mer,LNG,France,LNG
...,...,...,...,...,...,...,...,...,...,...,...
2026-03-27,120693468.0,entry,DE-TSO-0005,LNG-00060,BRUNSBUETTEL HAFEN (FSRU) (DE),DE,Liquefied Natural Gas,missing,LNG,Germany,LNG
2026-03-28,116337546.0,entry,DE-TSO-0005,LNG-00060,BRUNSBUETTEL HAFEN (FSRU) (DE),DE,Liquefied Natural Gas,missing,LNG,Germany,LNG
2026-03-29,118716855.0,entry,DE-TSO-0005,LNG-00060,BRUNSBUETTEL HAFEN (FSRU) (DE),DE,Liquefied Natural Gas,missing,LNG,Germany,LNG


In [89]:
# import LNG data from GIE (we only know where the LNG arrives, not where it comes from)
# agsi = pd.read_csv(r'C:\\Users\\giovanni.sgaravatti\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals\\raw_data\\agsi.csv') # Gio
agsi = pd.read_csv(r'C:\\Users\\ugne.keliauskaite\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals\\raw_data\\agsi.csv') # Ugne
agsi=agsi.set_index(pd.DatetimeIndex(agsi['dates']))
del agsi['dates']
del agsi['index']

In [90]:
# import Bloomberg LNG data (with these data we know both where it comes from and where it arrives, but we trust GIE better - also to be consistent with the tracker)
lng_b = pd.read_excel(r'LNG\Bloomberg\granular LNG imports.xlsx') # Gio
lng_b.rename(columns= {'Unnamed: 0':'dates'},inplace=True)
lng_b = lng_b.set_index(pd.DatetimeIndex(lng_b['dates']))
del lng_b['dates']

In [91]:
lng_b['Tot'] = lng_b.sum(axis=1,numeric_only=True)

In [92]:
# Divide each column by the 'Tot' column
ratios_df = lng_b.div(lng_b['Tot'], axis=0)

In [93]:
agsi_m = agsi.groupby(pd.Grouper(freq='M'))['sendOut'].sum(numeric_only=True)
# take only values after 2019 to be consistent with Bloomberg data
agsi_19 = agsi_m['2019':]

In [94]:
# to align with Bloombgerg lng
# agsi_19= agsi_19.iloc[:0] #Ugne change this one!

In [95]:
agsi_19

dates
2019-01-31     65357.1
2019-02-28     59014.4
2019-03-31     84049.6
2019-04-30     88014.5
2019-05-31     80457.8
                ...   
2025-11-30    130024.4
2025-12-31    133363.3
2026-01-31    133911.3
2026-02-28    123138.1
2026-03-31    136714.5
Freq: M, Name: sendOut, Length: 87, dtype: float64

In [96]:
# create new dataframe
lng = pd.DataFrame()
lng['dates'] = agsi_19.index

In [97]:
agsi_19
#agsi_19= agsi_19.iloc[:-1]

dates
2019-01-31     65357.1
2019-02-28     59014.4
2019-03-31     84049.6
2019-04-30     88014.5
2019-05-31     80457.8
                ...   
2025-11-30    130024.4
2025-12-31    133363.3
2026-01-31    133911.3
2026-02-28    123138.1
2026-03-31    136714.5
Freq: M, Name: sendOut, Length: 87, dtype: float64

In [98]:
#ratios_df= ratios_df.iloc[:-1] #Ugne change this one!

In [99]:
ratios_df

,Qatar,Algeria,Nigeria,Russia,Norway,United States,Trinidad & Tobago,Egypt,Other,Tot
dates,,,,,,,,,,
2019-01-31,0.227813,0.104325,0.187116,0.193431,0.046563,0.139817,0.074051,0.000000,0.026883,1.0
2019-02-28,0.195039,0.117453,0.146219,0.202086,0.062567,0.104679,0.076749,0.000000,0.095208,1.0
2019-03-31,0.217705,0.092288,0.139617,0.207924,0.054209,0.163316,0.068449,0.025039,0.031453,1.0
2019-04-30,0.174857,0.129060,0.138185,0.228259,0.052365,0.155742,0.085473,0.024848,0.011212,1.0
2019-05-31,0.220073,0.115558,0.135534,0.252179,0.054785,0.108252,0.056022,0.024447,0.033150,1.0
...,...,...,...,...,...,...,...,...,...,...
2025-11-30,0.054964,0.068855,0.056116,0.128108,0.039756,0.565386,0.021224,0.007679,0.057913,1.0
2025-12-31,0.097558,0.025028,0.051972,0.187230,0.037317,0.537274,0.016470,0.000000,0.047150,1.0
2026-01-31,0.045874,0.017402,0.044655,0.182871,0.043671,0.631340,0.007943,0.000000,0.026245,1.0


In [100]:
#lng = lng.iloc[:-1]

In [101]:
 # multiply Bloomberg LNG ratios by AGSI totals
# and convert to M3m
for column in ratios_df.columns:
    lng[column] = agsi_19.values*ratios_df[column].values/10.3

In [102]:
lng

,dates,Qatar,Algeria,Nigeria,Russia,Norway,United States,Trinidad & Tobago,Egypt,Other,Tot
0,2019-01-31,1445.554676,661.981080,1187.313715,1227.390108,295.455610,887.190554,469.879608,0.000000,170.584164,6345.349515
1,2019-02-28,1117.487058,672.951670,837.772109,1157.860659,358.479163,599.766228,439.738694,0.000000,545.497817,5729.553398
2,2019-03-31,1776.508485,753.087934,1139.297179,1696.694262,442.350493,1332.680771,558.551025,204.325134,256.660058,8160.155340
3,2019-04-30,1494.170233,1102.828446,1180.800058,1950.493874,447.462126,1330.831065,730.372421,212.329440,95.809425,8545.097087
4,2019-05-31,1719.089355,902.671484,1058.714685,1969.877584,427.952750,845.601098,437.613205,190.965978,258.950755,7811.436893
...,...,...,...,...,...,...,...,...,...,...,...
82,2025-11-30,693.847118,869.202352,708.389719,1617.203052,501.875226,7137.277839,267.927275,96.933127,731.072449,12623.728155
83,2025-12-31,1263.170427,324.063797,672.933888,2424.233568,483.181492,6956.569337,213.249960,0.000000,610.490734,12947.893204
84,2026-01-31,596.405980,226.248392,580.568950,2377.518902,567.775681,8208.107303,103.262137,0.000000,341.209743,13001.097087
85,2026-02-28,705.493828,197.523108,995.100221,1893.056813,511.258212,6922.143256,90.393287,45.262757,594.923858,11955.155340


In [103]:
lng.set_index(pd.DatetimeIndex(lng['dates']),inplace=True)
del lng['dates']

In [104]:
lng['Total less Russia and USA and NO and AL'] = lng['Tot'] - lng['Russia'] - lng['United States'] - lng['Norway'] - lng['Algeria']

In [105]:
# Change the months for which you have data here
months = pd.date_range(start='2019-01-01', end='2026-03-31', freq='M')

In [106]:
entsog = entsog['2019':]

In [107]:
converter = 10300000   ## KWh to M3m --> 10.3 KWh/m^3     # on ENTSOG/AGSI the data comes in KWh, we transform it (later on) in M3m 

In [108]:
entsog_m = pd.DataFrame()
entsog_m['dates'] = months
entsog_m.set_index(pd.DatetimeIndex(entsog_m['dates']),inplace=True)
del entsog_m['dates']

for country in ['Russia', 'Norway','Algeria', 'UK', 'Azerbaijan','Libya']:
    entsog_m[country] = entsog[entsog['aggregation'] == country]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [109]:
for pipe in ['Ukraine Gas Transit', 'Yamal (BY,PL)','Nord Stream', 'Turkstream']:
    entsog_m[pipe] = entsog[entsog['aggregation2'] == pipe]['values'].groupby(pd.Grouper(freq='M')).sum(numeric_only=True)/converter

In [110]:
entsog_q = entsog_m.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)
lng_q = lng.groupby(pd.Grouper(freq='Q')).sum(numeric_only=True)

In [111]:
graph = entsog_q
graph['USA LNG'] = lng_q['United States']
graph['Russia LNG'] = lng_q['Russia']
graph['Norway LNG'] = lng_q['Norway']
graph['Algeria LNG'] = lng_q['Algeria']
del graph['Russia']
graph['LNG less RU and USA and NO and AL'] = lng_q['Total less Russia and USA and NO and AL']
graph = graph['2019':]

In [112]:
graph.tail()

,Norway,Algeria,UK,Azerbaijan,Libya,Ukraine Gas Transit,"Yamal (BY,PL)",Nord Stream,Turkstream,USA LNG,Russia LNG,Norway LNG,Algeria LNG,LNG less RU and USA and NO and AL
dates,,,,,,,,,,,,,,
2025-03-31,22526.393024,8392.572365,1939.164978,2839.859508,233.770742,0.0,0.0,0.0,4531.714854,18428.668866,5491.677576,1067.687419,1403.809181,8089.933656
2025-06-30,23945.617855,8027.203807,4523.536607,3045.624929,298.820568,0.0,0.0,0.0,3817.175859,22335.429484,5571.595855,463.711280,2222.421607,7672.074783
2025-09-30,23040.787093,7081.804077,5130.792579,3201.157704,105.709883,0.0,0.0,0.0,4716.390557,20526.179338,3433.464091,881.496952,1839.330291,5927.772047
2025-12-31,23881.824271,7538.332311,2495.716473,3254.447317,343.077156,0.0,0.0,0.0,5065.132273,21647.872814,5365.023233,1407.349794,2099.577020,7834.817916
2026-03-31,23662.079627,8624.044620,2540.786736,2989.999921,106.926997,0.0,0.0,0.0,4976.374300,22683.464160,6543.565494,1526.790660,1112.430126,6363.254415


In [113]:
graph = graph[['Nord Stream','Yamal (BY,PL)','Ukraine Gas Transit','Turkstream','Russia LNG', 'Norway LNG', 'USA LNG', 'Algeria LNG', 'LNG less RU and USA and NO and AL', 'Norway','Algeria','UK','Azerbaijan','Libya']]

In [114]:
from datetime import datetime
today = date.today()

In [115]:
with pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today)) as writer:
    Excelwriter = pd.ExcelWriter("Other\quarterly_data {}.xlsx".format(today),engine="xlsxwriter")
    entsog_q.to_excel(Excelwriter, sheet_name="ENTSOG", index=True)
    lng.to_excel(Excelwriter, sheet_name="LNG", index=True)
    graph.to_excel(Excelwriter, sheet_name="graph", index=True)
Excelwriter.close()
Excelwriter.save()

C:\Users\ugne.keliauskaite\AppData\Local\Temp\ipykernel_84240\2191511067.py:7: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  Excelwriter.save()
c:\Users\ugne.keliauskaite\AppData\Local\anaconda3\Lib\site-packages\xlsxwriter\workbook.py:368: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")
